## Naive Bayes from Scratch: Network Access Screening



## Dataset

Each row describes one network access record. Every feature is binary: `1` means the signal is present and `0` means it is absent.

| Signal | Meaning |
|---|---|
| `failed_logins` | Multiple failed login attempts occurred. |
| `unusual_port` | The connection used an unusual port. |
| `large_transfer` | A large outbound transfer occurred. |
| `new_device` | The device has not been observed before. |
| `encrypted_payload` | The payload could not be inspected. |
| `off_hours` | The access occurred outside normal working hours. |

The target is `access_type`, and `subset` provides a fixed split.

In [1]:
%pip install numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

access_columns = [
    "failed_logins",
    "unusual_port",
    "large_transfer",
    "new_device",
    "encrypted_payload",
    "off_hours",
    "access_type",
    "subset",
]

access_rows = [
    [1, 1, 1, 0, 1, 1, "intrusion", "train"],
    [1, 1, 0, 1, 1, 1, "intrusion", "train"],
    [1, 0, 1, 1, 1, 0, "intrusion", "train"],
    [0, 1, 1, 1, 1, 1, "intrusion", "train"],
    [1, 1, 0, 0, 1, 0, "intrusion", "train"],
    [0, 1, 1, 1, 0, 1, "intrusion", "train"],
    [1, 0, 1, 0, 1, 1, "intrusion", "train"],
    [1, 1, 1, 1, 1, 1, "intrusion", "train"],
    [0, 1, 0, 1, 1, 1, "intrusion", "train"],
    [1, 1, 1, 0, 0, 1, "intrusion", "train"],
    [1, 0, 0, 1, 1, 1, "intrusion", "train"],
    [0, 1, 1, 0, 1, 0, "intrusion", "train"],
    [0, 0, 0, 0, 0, 0, "legitimate", "train"],
    [0, 0, 1, 0, 0, 0, "legitimate", "train"],
    [0, 0, 0, 1, 0, 0, "legitimate", "train"],
    [1, 0, 0, 0, 0, 0, "legitimate", "train"],
    [0, 1, 0, 0, 0, 0, "legitimate", "train"],
    [0, 0, 0, 0, 1, 0, "legitimate", "train"],
    [0, 0, 0, 1, 0, 1, "legitimate", "train"],
    [0, 1, 0, 0, 0, 0, "legitimate", "train"],
    [0, 0, 1, 0, 0, 1, "legitimate", "train"],
    [1, 0, 0, 1, 0, 0, "legitimate", "train"],
    [0, 0, 0, 0, 0, 1, "legitimate", "train"],
    [0, 0, 1, 1, 0, 0, "legitimate", "train"],
    [1, 1, 1, 1, 1, 1, "intrusion", "test"],
    [1, 0, 1, 1, 1, 0, "intrusion", "test"],
    [0, 1, 1, 0, 1, 1, "intrusion", "test"],
    [1, 1, 0, 1, 0, 1, "intrusion", "test"],
    [0, 0, 0, 0, 0, 0, "legitimate", "test"],
    [0, 0, 1, 0, 0, 1, "legitimate", "test"],
    [1, 0, 0, 1, 0, 0, "legitimate", "test"],
    [0, 1, 0, 0, 0, 0, "legitimate", "test"],
]

access_log = pd.DataFrame(access_rows, columns=access_columns)
access_log

,failed_logins,unusual_port,large_transfer,new_device,encrypted_payload,off_hours,access_type,subset
0,1,1,1,0,1,1,intrusion,train
1,1,1,0,1,1,1,intrusion,train
2,1,0,1,1,1,0,intrusion,train
3,0,1,1,1,1,1,intrusion,train
4,1,1,0,0,1,0,intrusion,train
5,0,1,1,1,0,1,intrusion,train
6,1,0,1,0,1,1,intrusion,train
7,1,1,1,1,1,1,intrusion,train
8,0,1,0,1,1,1,intrusion,train
9,1,1,1,0,0,1,intrusion,train


In [3]:
security_signals = [
    "failed_logins",
    "unusual_port",
    "large_transfer",
    "new_device",
    "encrypted_payload",
    "off_hours",
]

known_accesses = access_log[access_log["subset"] == "train"].reset_index(drop=True)
unknown_accesses = access_log[access_log["subset"] == "test"].reset_index(drop=True)

signal_train = known_accesses[security_signals]
type_train = known_accesses["access_type"]
signal_test = unknown_accesses[security_signals]
type_test = unknown_accesses["access_type"]
class_order = np.sort(type_train.unique())

print("Known accesses:  ", signal_train.shape)
print("Unknown accesses:", signal_test.shape)
print(type_train.value_counts())

Known accesses:   (24, 6)
Unknown accesses: (8, 6)
access_type
intrusion     12
legitimate    12
Name: count, dtype: int64


## Ordered prior vector 

Complete `derive_prior_vector`. It must return both:

1. `label_totals`: a dictionary containing the training count of each class.
2. `prior_vector`: a NumPy array of priors in exactly the same order as `class_order`.

In [4]:
def derive_prior_vector(targets, ordered_classes):
    """Calculate class totals and an ordered vector of priors.

    Parameters
    ----------
    targets : pandas.Series
        Training class labels.
    ordered_classes : array-like
        Required order of classes in the prior vector.

    Returns
    -------
    tuple
        A count dictionary and a one-dimensional NumPy prior array.
    """
    label_totals = {}
    ordered_priors = []

    for access_class in ordered_classes:
        class_total = (targets == access_class).sum()
        class_prior = class_total / len(targets)
        label_totals[access_class] = class_total
        ordered_priors.append(class_prior)

    return label_totals, np.asarray(ordered_priors, dtype=float)


label_totals, prior_vector = derive_prior_vector(type_train, class_order)
print(label_totals)
print(prior_vector)

{'intrusion': np.int64(12), 'legitimate': np.int64(12)}
[0.5 0.5]


## Signal probabilities 

Instead of storing separate probabilities for values 0 and 1, estimate only the probability that each signal equals 1:

$$\theta_{j,c}=P(X_j=1\mid C=c)=\frac{N_{j,1,c}+\alpha}{N_c+2\alpha}$$

Complete `fit_one_probabilities`. Return a DataFrame whose rows are signal names and whose columns follow `class_order`. Use $\alpha=1$.

In [5]:
def fit_one_probabilities(binary_inputs, targets, ordered_classes, alpha=1.0):
    """Estimate P(signal=1 | class) for binary features.

    Parameters
    ----------
    binary_inputs : pandas.DataFrame
        Binary training signals.
    targets : pandas.Series
        Training labels aligned with binary_inputs.
    ordered_classes : array-like
        Required class-column order.
    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    pandas.DataFrame
        Signal rows and class columns containing P(signal=1 | class).
    """
    one_probabilities = pd.DataFrame(
        index=binary_inputs.columns,
        columns=ordered_classes,
        dtype=float,
    )

    for access_class in ordered_classes:
        class_signals = binary_inputs[targets == access_class]

        for signal_name in binary_inputs.columns:
            one_count = (class_signals[signal_name] == 1).sum()
            one_probability = (one_count + alpha) / (len(class_signals) + 2 * alpha)
            one_probabilities.loc[signal_name, access_class] = one_probability

    return one_probabilities


signal_one_probabilities = fit_one_probabilities(
    signal_train, type_train, class_order, alpha=1.0
)
signal_one_probabilities

,intrusion,legitimate
failed_logins,0.642857,0.214286
unusual_port,0.714286,0.214286
large_transfer,0.642857,0.285714
new_device,0.571429,0.357143
encrypted_payload,0.785714,0.142857
off_hours,0.714286,0.285714


## Binary-feature log-score matrix

Complete `access_score_matrix`. For a binary record $\mathbf{x}$, use both the present-signal and absent-signal contributions:

$$L(c)=\log P(c)+\sum_j[x_j\log\theta_{j,c}+(1-x_j)\log(1-\theta_{j,c})]$$

Then create `security_predictions` as a Pandas Series named `predicted_access`, using the largest score in each row.

In [6]:
def access_score_matrix(
    observations, ordered_classes, ordered_priors, one_probabilities
):
    """Calculate Naive Bayes log-scores for binary signals.

    Parameters
    ----------
    observations : pandas.DataFrame
        Binary records to score.
    ordered_classes : array-like
        Class labels matching ordered_priors.
    ordered_priors : numpy.ndarray
        Class prior probabilities.
    one_probabilities : pandas.DataFrame
        P(signal=1 | class) with signal rows and class columns.

    Returns
    -------
    pandas.DataFrame
        Class log-scores for every observation.
    """
    score_values = np.zeros((len(observations), len(ordered_classes)))

    for class_position, access_class in enumerate(ordered_classes):
        probabilities_of_one = one_probabilities[access_class].to_numpy()

        for row_position, (_, observation) in enumerate(observations.iterrows()):
            signal_values = observation.to_numpy()

            present_contribution = np.log(probabilities_of_one) * signal_values
            absent_contribution = np.log(1 - probabilities_of_one) * (1 - signal_values)
            score_values[row_position, class_position] = np.log(ordered_priors[class_position]) 
            + present_contribution.sum() + absent_contribution.sum()

    return pd.DataFrame(
        score_values, index=observations.index, columns=ordered_classes
    )


security_scores = access_score_matrix(
    signal_test, class_order, prior_vector, signal_one_probabilities
)

security_predictions = security_scores.idxmax(axis=1)

security_report = signal_test.copy()
security_report["actual_access"] = type_test.to_numpy()
security_report["predicted_access"] = security_predictions.to_numpy()
display(security_scores)
display(security_report)

,intrusion,legitimate
0,-0.693147,-0.693147
1,-0.693147,-0.693147
2,-0.693147,-0.693147
3,-0.693147,-0.693147
4,-0.693147,-0.693147
5,-0.693147,-0.693147
6,-0.693147,-0.693147
7,-0.693147,-0.693147


,failed_logins,unusual_port,large_transfer,new_device,encrypted_payload,off_hours,actual_access,predicted_access
0,1,1,1,1,1,1,intrusion,intrusion
1,1,0,1,1,1,0,intrusion,intrusion
2,0,1,1,0,1,1,intrusion,intrusion
3,1,1,0,1,0,1,intrusion,intrusion
4,0,0,0,0,0,0,legitimate,intrusion
5,0,0,1,0,0,1,legitimate,intrusion
6,1,0,0,1,0,0,legitimate,intrusion
7,0,1,0,0,0,0,legitimate,intrusion


## Security audit metrics 

Complete `create_security_audit` without using a metrics library. Rows must represent **predicted** classes and columns must represent **actual** classes. Then calculate:

- `detection_accuracy`: the fraction of all correct predictions.
- `false_alarm_rate`: the fraction of actual `legitimate` records predicted as `intrusion`.

In [7]:
def create_security_audit(actual_types, predicted_types, ordered_classes):
    """Build an audit table with predicted rows and actual columns.

    Parameters
    ----------
    actual_types : array-like
        Actual access classes.
    predicted_types : array-like
        Predicted access classes.
    ordered_classes : array-like
        Required row and column class order.

    Returns
    -------
    pandas.DataFrame
        Count table with predicted rows and actual columns.
    """
    audit_counts = np.zeros(
        (len(ordered_classes), len(ordered_classes)), dtype=int
    )
    class_position = {
        access_class: position
        for position, access_class in enumerate(ordered_classes)
    }

    for actual, predicted in zip(actual_types, predicted_types):
        row = class_position[predicted]
        col = class_position[actual]
        audit_counts[row, col] += 1
    

    return pd.DataFrame(
        audit_counts,
        index=[f"predicted_{name}" for name in ordered_classes],
        columns=[f"actual_{name}" for name in ordered_classes],
    )


security_audit = create_security_audit(
    type_test, security_predictions, class_order
)

detection_accuracy = np.diag(security_audit).sum() / security_audit.to_numpy().sum()
false_alarm_rate = 1 - (security_audit.loc["predicted_intrusion", "actual_intrusion"] / 
                        security_audit.loc["predicted_intrusion"].sum())

print(f"Detection accuracy: {detection_accuracy:.3f}")
print(f"False-alarm rate:  {false_alarm_rate:.3f}")
display(security_audit)

Detection accuracy: 0.500
False-alarm rate:  0.500


,actual_intrusion,actual_legitimate
predicted_intrusion,4,4
predicted_legitimate,0,0
